# Evaluation Summary

This notebook summarizes the evaluation results of the trained models.

## 1. Import Libraries

In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import pyarrow.parquet as pq
from ipywidgets import interact, FloatSlider


## 2. Define Paths and Load Configurations

In [ ]:
# Define the parent directory for experiments
output_parent_dir = os.path.abspath(os.path.expanduser(
    '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption/experiments/nasa_milling_1dconv'
))

# Load configs and model paths from all experiments
experiment_data = []
config_dir = os.path.join(output_parent_dir, 'configs')
model_dir = os.path.join(output_parent_dir, 'models')

if os.path.exists(model_dir) and os.path.exists(config_dir):
    # Create a dictionary mapping timestamps to config file paths
    config_files = {
        f.split('config_')[-1].replace('.yaml', ''): os.path.join(config_dir, f)
        for f in os.listdir(config_dir) if f.endswith('.yaml')
    }
    
    # Find all model files and match them with configs
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.keras')]
    for model_filename in model_files:
        model_path = os.path.join(model_dir, model_filename)
        # Extract timestamp from filename, e.g., 1d_conv_model_20251009_090843.keras
        timestamp = model_filename.split('model_')[-1].replace('.keras', '')
        
        if timestamp in config_files:
            config_path = config_files[timestamp]
            with open(config_path, 'r') as f:
                config = yaml.safe_load(f)
            
            experiment_data.append({
                'config_path': config_path,
                'model_path': model_path,
                'config': config,
                'train_caseIDs': config.get('train_caseIDs', []),
                'test_caseIDs': config.get('test_caseIDs', [])
            })

# Create a DataFrame to display the experiment configurations
if experiment_data:
    experiments_df = pd.DataFrame([{
        'Train Cases': d['train_caseIDs'],
        'Test Cases': d['test_caseIDs'],
        'Model Path': os.path.basename(d['model_path']),
        'Config Path': os.path.basename(d['config_path'])
    } for d in experiment_data])
else:
    experiments_df = pd.DataFrame()

# Sort by train and test cases for consistent order
if not experiments_df.empty:
    # Convert list to tuple for sorting
    experiments_df['Train Cases'] = experiments_df['Train Cases'].apply(tuple)
    experiments_df['Test Cases'] = experiments_df['Test Cases'].apply(tuple)
    experiments_df = experiments_df.sort_values(by=['Train Cases', 'Test Cases']).reset_index(drop=True)

experiments_df

In [ ]:
#get all unique case IDs from all experiments
all_case_ids = set()
for data in experiment_data:
    all_case_ids.update(data['train_caseIDs'])
    all_case_ids.update(data['test_caseIDs'])
all_case_ids = sorted(all_case_ids)
print(all_case_ids)

## 3. Load and Prepare Test Data

In [ ]:
signal_columns = ['vib_spindle']

In [ ]:
# merge single experiment results into one dataframe
results_df = pd.DataFrame()
# The single files are in the logs folder named: evaluation_results_1d_conv_$timestamp.csv but just get every csv file in the folder
# They all have the same header so we can just concatenate them
logs_dir = os.path.join(output_parent_dir, 'logs')
if os.path.exists(logs_dir):
    log_files = [f for f in os.listdir(logs_dir) if f.endswith('.csv')]
    for log_file in log_files:
        log_path = os.path.join(logs_dir, log_file)
        df = pd.read_csv(log_path)
        results_df = pd.concat([results_df, df], ignore_index=True)

In [ ]:
display(results_df)

In [ ]:
def load_data(input_path):
    """
    Loads data from a Parquet file with large array columns memory-efficiently.
    It reads metadata columns first, then loads large signal columns one by one.
    """
    print(f"Memory-efficiently loading data from {input_path}...")
    
    try:
        parquet_file = pq.ParquetFile(input_path)
        
        # Get all column names from the file's schema
        all_columns = parquet_file.schema.names
        
        # Identify metadata columns (all columns that are NOT signals)
        signal_cols = signal_columns
        metadata_cols = [col for col in all_columns if col not in signal_cols]
        
        # 1. Load only the small metadata columns first
        print(f"Loading metadata columns: {metadata_cols}")
        if metadata_cols:
            dataDF = parquet_file.read(columns=metadata_cols).to_pandas()
        else:
            # If no metadata, create an empty DataFrame with the right number of rows
            num_rows = parquet_file.metadata.num_rows
            dataDF= pd.DataFrame(index=range(num_rows))

        # 2. Load each large signal column individually and add it to the DataFrame
        for col in signal_cols:
            print(f"Loading signal column: {col}")
            # read_row_group can also be used for more granular control if needed
            signal_series = parquet_file.read(columns=[col]).to_pandas()[col]
            dataDF[col] = signal_series

        print(f"Successfully loaded {len(dataDF)} rows and columns: {dataDF.columns.tolist()}")

    except Exception as e:
        print(f"An error occurred during data loading: {e}")
        dataDF = None
    
    return dataDF


def set_wear_th(data_df, wear_th, wear_column_name):
    data_df['wear_class'] = np.where(data_df[wear_column_name] > wear_th, 1, 0)
    return data_df

def reformat_data(data_df, signal_columns, label_column):
    sequence_data = [
        np.stack([row[col] for col in signal_columns], axis=-1)
        for _, row in tqdm.tqdm(data_df.iterrows(), total=len(data_df), desc='Extracting sequences')
    ]
    labels = data_df[label_column].to_numpy()
    return sequence_data, labels

def rectangular_sequence_data(sequence_data, signal_length, pooling_type='mean'):
    num_samples = len(sequence_data)
    if num_samples == 0:
        return np.array([])
    num_signals = sequence_data[0].shape[1]
    rectangular = np.zeros((num_samples, signal_length, num_signals), dtype=sequence_data[0].dtype)
    for i, seq in enumerate(tqdm.tqdm(sequence_data, desc='Processing sequences')):
        seq_len = seq.shape[0]
        if seq_len > signal_length:
            # Downsample using pooling
            pool_size = int(np.ceil(seq_len / signal_length))
            if pooling_type == 'mean':
                pooled_seq = np.array([np.mean(seq[i:i+pool_size], axis=0) for i in range(0, seq_len, pool_size)])
            elif pooling_type == 'max':
                pooled_seq = np.array([np.max(seq[i:i+pool_size], axis=0) for i in range(0, seq_len, pool_size)])
            else:
                raise ValueError(f"Unknown pooling type: {pooling_type}")
            rectangular[i, :min(len(pooled_seq), signal_length)] = pooled_seq[:signal_length]
        else:
            # Pad with zeros
            rectangular[i, :seq_len] = seq
    return rectangular

def normalize_data(X_train, X_test):
    X_train_np = np.array(X_train, dtype=np.float32)
    X_test_np = np.array(X_test, dtype=np.float32)
    
    nsamples, nx, ny = X_train_np.shape
    X_train_2d = X_train_np.reshape((nsamples * nx, ny))

    scaler = StandardScaler()
    scaler.fit(X_train_2d)

    X_train_scaled_2d = scaler.transform(X_train_2d)
    X_train_scaled = X_train_scaled_2d.reshape(nsamples, nx, ny)

    nsamples_test, nx_test, ny_test = X_test_np.shape
    X_test_2d = X_test_np.reshape((nsamples_test * nx_test, ny_test))
    X_test_scaled_2d = scaler.transform(X_test_2d)
    X_test_scaled = X_test_scaled_2d.reshape(nsamples_test, nx_test, ny_test)
    
    return X_train_scaled, X_test_scaled, scaler

def prepare_test_data(config, is_autoencoder=False):
    data_path = config['inputPath']
    if not os.path.exists(data_path):
        print(f"Data file not found at: {data_path}")
        try:
            project_root = '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption'
            relative_path_part = config['inputPath'].split('tool-wear-domain-adaption/')[-1]
            new_data_path = os.path.join(project_root, relative_path_part)
            
            if os.path.exists(new_data_path):
                print(f"Found data at alternative path: {new_data_path}")
                data_path = new_data_path
            else:
                 print(f"Alternative path also not found: {new_data_path}")
                 return None, None
        except Exception as e:
            print(f"Could not construct an alternative path: {e}")
            return None, None

    data_df = load_data(data_path)
    if data_df is None:
        return None, None
        
    data_df = set_wear_th(data_df, config['wearTH'], config['wearColumnName'])
    
    train_df = data_df[data_df['CaseID'].isin(config['train_caseIDs'])].copy()
    test_df = data_df[data_df['CaseID'].isin(config['test_caseIDs'])].copy()

    # For autoencoders, the training data for the scaler should only be healthy data
    if is_autoencoder:
        # In training, autoencoder was trained on healthy data only.
        # We mimic this for the scaler by fitting it only on healthy data from the training set.
        train_df = train_df[train_df[config['labelColumn']] == 0]

    # Prepare training data to fit the scaler
    sequence_train_raw, _ = reformat_data(train_df, config['signalColumns'], config['labelColumn'])
    sequence_train = rectangular_sequence_data(sequence_train_raw, config['signal_length'], config['pooling_type'])

    # Prepare test data
    sequence_test_raw, labels_test = reformat_data(test_df, config['signalColumns'], config['labelColumn'])
    sequence_test = rectangular_sequence_data(sequence_test_raw, config['signal_length'], config['pooling_type'])
    
    # Apply autoencoder-specific preprocessing if specified
    if is_autoencoder:
        # Downsample, clip and scale
        downsample_rate = config.get('downsample_rate', 1)
        clip_range = config.get('clip_range', None)

        sequence_train = sequence_train[:, ::downsample_rate, :]
        sequence_test = sequence_test[:, ::downsample_rate, :]
        
        if clip_range:
            sequence_train = np.clip(sequence_train, clip_range[0], clip_range[1])
            sequence_test = np.clip(sequence_test, clip_range[0], clip_range[1])

    # Explicitly set num_classes=2 to handle cases where test set has only one class
    labels_test_cat = to_categorical(labels_test, num_classes=2)

    # Normalize test data using a scaler fitted on the training data
    _, X_test_scaled, _ = normalize_data(sequence_train, sequence_test)
    
    return X_test_scaled, labels_test_cat

# Example of preparing data for the first experiment
if experiment_data:
    first_exp_config = experiment_data[0]['config']
    is_autoencoder_example = 'autoenc' in output_parent_dir
    X_test_example, y_test_example = prepare_test_data(first_exp_config, is_autoencoder=is_autoencoder_example)
    if X_test_example is not None:
        print(f"Shape of prepared test data (X): {X_test_example.shape}")
        print(f"Shape of prepared test labels (y): {y_test_example.shape}")
else:
    print("No experiment data found.")

## 4. Evaluate Models and Collect Results

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
from IPython.display import display, Markdown

results = []
is_autoencoder = 'autoenc' in output_parent_dir

for exp_data in tqdm.tqdm(experiment_data, desc="Evaluating models"):
    # Load model from the directory path
    model = tf.keras.models.load_model(exp_data['model_path'])
    
    # Prepare data for this specific experiment's configuration
    X_test, y_test = prepare_test_data(exp_data['config'], is_autoencoder=is_autoencoder)
    
    if is_autoencoder:
        # For autoencoders, evaluation is based on reconstruction error
        reconstructions = model.predict(X_test)
        # Calculate MAE and then average over the time steps (axis=1)
        recon_error_per_step = tf.keras.losses.mae(reconstructions, X_test)
        recon_error = tf.reduce_mean(recon_error_per_step, axis=1)
        y_true_classes = np.argmax(y_test, axis=1)
        
        # Here we just store the raw errors and true labels.
        # The thresholding and metric calculation will be done in the interactive cell.
        results.append({
            'train_caseIDs': exp_data['train_caseIDs'],
            'test_caseIDs': exp_data['test_caseIDs'],
            'reconstruction_error': recon_error.numpy(),
            'y_true': y_true_classes
        })
        continue # Skip the rest of the loop for autoencoders

    # Evaluate model
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    
    # Get predictions
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)
    
    # Calculate confusion matrix
    cm = confusion_matrix(y_true_classes, y_pred_classes)
    
    # Calculate precision, recall, and f1-score using scikit-learn for robustness
    # zero_division=0 sets the metric to 0 if there are no true positives.
    precision = precision_score(y_true_classes, y_pred_classes, zero_division=0)
    recall = recall_score(y_true_classes, y_pred_classes, zero_division=0)
    f1 = f1_score(y_true_classes, y_pred_classes, zero_division=0)
    
    results.append({
        'train_caseIDs': exp_data['train_caseIDs'],
        'test_caseIDs': exp_data['test_caseIDs'],
        'accuracy': accuracy,
        'loss': loss,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm,
        'y_true': y_true_classes,
        'y_pred': y_pred_classes
    })

# Create a DataFrame to display the results
results_df = pd.DataFrame(results)

if not is_autoencoder:
    # Rename columns for clarity and consistency
    results_df = results_df.rename(columns={
        'train_caseIDs': 'Train Cases',
        'test_caseIDs': 'Test Cases',
        'accuracy': 'Accuracy',
        'loss': 'Loss',
        'precision': 'Precision',
        'recall': 'Recall',
        'f1_score': 'F1-Score',
        'y_true': 'y_true_classes',
        'y_pred': 'y_pred_classes'
    })

    # Reorder columns to have metrics first, then the data arrays
    desired_columns = [
        'Train Cases', 'Test Cases', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'Loss',
        'confusion_matrix', 'y_true_classes', 'y_pred_classes'
    ]
    results_df = results_df[desired_columns]


    # Sort by train and test cases for consistent order
    if not results_df.empty:
        # Convert list to tuple for sorting
        results_df['Train Cases'] = results_df['Train Cases'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
        results_df['Test Cases'] = results_df['Test Cases'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
        results_df = results_df.sort_values(by=['Train Cases', 'Test Cases']).reset_index(drop=True)

    display(Markdown("### Overall Performance Metrics"))

display(results_df)

In [ ]:
# Save if not already performed during batch evaluation
summary_path = os.path.join(output_parent_dir, 'evaluation_summary.csv')
if not os.path.exists(summary_path):
    results_df.to_csv(summary_path, index=False)
    print(f"Saved evaluation summary to {summary_path}")

## 5. Visualize Confusion Matrices

In [ ]:
def plot_confusion_matrices(results):
    num_results = len(results)
    if num_results == 0:
        print("No results to display.")
        return
    
    # Determine grid size
    cols = 3
    rows = (num_results + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = axes.flatten() # Flatten to 1D array for easy iteration

    for i, result in enumerate(results):
        disp = ConfusionMatrixDisplay(confusion_matrix=result['confusion_matrix'])
        disp.plot(cmap=plt.cm.Blues, ax=axes[i])
        title = f"Train: {result['train_caseIDs']}\\nTest: {result['test_caseIDs']}\\nAcc: {result['accuracy']:.2f}"
        axes[i].set_title(title)

    # Hide unused subplots
    for i in range(num_results, len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.show()

plot_confusion_matrices(results)

## 6. Accuracy Heatmap

In [ ]:
def plot_accuracy_heatmap(results_df):
    if results_df.empty:
        print("No results to display.")
        return

    # The 'Train Cases' and 'Test Cases' are tuples, convert them to strings for plotting
    results_df['Train Cases Str'] = results_df['Train Cases'].astype(str)
    results_df['Test Cases Str'] = results_df['Test Cases'].astype(str)

    # Create a pivot table for the heatmap
    try:
        # Use pivot_table to handle duplicate entries by averaging them
        heatmap_data = results_df.pivot_table(index='Train Cases Str', columns='Test Cases Str', values='Accuracy', aggfunc='mean')
        
        plt.figure(figsize=(12, 8))
        sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="viridis")
        plt.title('Model Accuracy Heatmap (Mean Accuracy)')
        plt.xlabel('Test Case IDs')
        plt.ylabel('Train Case IDs')
        plt.show()
    except Exception as e:
        print(f"Could not create heatmap. Error: {e}")
        print("Data might not be in the right format for pivoting.")


plot_accuracy_heatmap(results_df)

## 7. Bar Chart of Accuracy per Experiment

In [ ]:
def plot_metrics_bar_chart(results_df):
    if results_df.empty:
        print("No results to display.")
        return

    # Create a descriptive label for each experiment
    results_df['Experiment'] = results_df.apply(
        lambda row: f"Train: {row['Train Cases']}\\nTest: {row['Test Cases']}", 
        axis=1
    )

    # Melt the DataFrame to have metrics in a single column for hue
    metrics_df = results_df.melt(
        id_vars=['Experiment'], 
        value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score'],
        var_name='Metric', 
        value_name='Score'
    )

    plt.figure(figsize=(15, 10))
    ax = sns.barplot(x='Score', y='Experiment', hue='Metric', data=metrics_df, orient='h')
    
    plt.title('Model Performance Metrics for Each Experiment')
    plt.xlabel('Score')
    plt.ylabel('Experiment Configuration')
    plt.xlim(0, 1.15) # Extend x-axis for legend and labels
    plt.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')

    # Add the score labels on each bar
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', label_type='edge', padding=2)

    plt.tight_layout()
    plt.show()

plot_metrics_bar_chart(results_df)

## 9. Wear Distribution Analysis

In [ ]:

def load_full_data_for_exp(config):
    """
    Loads the full DataFrame and splits it into train and test sets based on CaseIDs.
    """
    data_path = config['inputPath']

    if not os.path.exists(data_path):
        print(f"Data file not found at: {data_path}")
        # Attempt to construct a relative path from the project root, assuming a standard structure
        # This is a fallback if the absolute path from the config is incorrect in the current environment.
        try:
            # Assumes the notebook is in a subfolder of the project root
            project_root = '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption'
            # The original config seems to have the full path, but let's try to be robust
            # The inputPath is an absolute path, but let's try to construct one if it fails
            relative_path_part = config['inputPath'].split('tool-wear-domain-adaption/')[-1]
            new_data_path = os.path.join(project_root, relative_path_part)
            
            if os.path.exists(new_data_path):
                print(f"Found data at alternative path: {new_data_path}")
                data_path = new_data_path
            else:
                 print(f"Alternative path also not found: {new_data_path}")
                 return None, None
        except Exception as e:
            print(f"Could not construct an alternative path: {e}")
            return None, None


    data_df = load_data(data_path)
    if data_df is None:
        return None, None

    train_df = data_df[data_df['CaseID'].isin(config['train_caseIDs'])].copy()
    test_df = data_df[data_df['CaseID'].isin(config['test_caseIDs'])].copy()
    
    return train_df, test_df

def plot_wear_distribution(experiment_data):
    if not experiment_data:
        print("No experiment data to display.")
        return

    num_results = len(experiment_data)
    cols = 2  # Plot train and test side-by-side
    rows = num_results
    
    fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows), sharex=True)
    if rows == 1:
        axes = np.array([axes]) # Ensure axes is always a 2D array

    for i, exp_data in enumerate(experiment_data):
        config = exp_data['config']
        train_df, test_df = load_full_data_for_exp(config)

        ax_train = axes[i, 0]
        ax_test = axes[i, 1]

        if train_df is not None and not train_df.empty:
            sns.histplot(data=train_df, x='wear_norm', ax=ax_train, kde=True)
            ax_train.set_title(f"Train Cases: {config['train_caseIDs']}")
            ax_train.set_xlabel("Normalized Wear (`wear_norm`)")
            ax_train.set_ylabel("Count")
        else:
            ax_train.set_title(f"Train Cases: {config['train_caseIDs']} (Data not found or empty)")
            ax_train.set_visible(False)

        if test_df is not None and not test_df.empty:
            sns.histplot(data=test_df, x='wear_norm', ax=ax_test, kde=True)
            ax_test.set_title(f"Test Cases: {config['test_caseIDs']}")
            ax_test.set_xlabel("Normalized Wear (`wear_norm`)")
            ax_test.set_ylabel("Count")
        else:
            ax_test.set_title(f"Test Cases: {config['test_caseIDs']} (Data not found or empty)")
            ax_test.set_visible(False)


    plt.suptitle('Distribution of Normalized Wear for Train and Test Sets', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

# Use the 'experiment_data' loaded in cell 5, which is based on the trained models.
# This ensures we are analyzing the exact configurations that were trained and evaluated.
plot_wear_distribution(experiment_data)


## 10. Wear Class Distribution Analysis

In [ ]:

def load_wear_data_for_exp(config):
    """
    Loads only the wear-related and CaseID columns from the dataset for a specific experiment.
    """
    data_path = config['inputPath']
    wear_column_name = config['wearColumnName']
    
    if not os.path.exists(data_path):
        print(f"Data file not found at: {data_path}")
        try:
            project_root = '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption'
            relative_path_part = config['inputPath'].split('tool-wear-domain-adaption/')[-1]
            new_data_path = os.path.join(project_root, relative_path_part)
            
            if os.path.exists(new_data_path):
                print(f"Found data at alternative path: {new_data_path}")
                data_path = new_data_path
            else:
                 print(f"Alternative path also not found: {new_data_path}")
                 return None, None
        except Exception as e:
            print(f"Could not construct an alternative path: {e}")
            return None, None

    try:
        parquet_file = pq.ParquetFile(data_path)
        # We need CaseID for filtering and the wear column for the plot.
        columns_to_load = ['CaseID', wear_column_name]
        
        # Check if columns exist before trying to load
        available_columns = parquet_file.schema.names
        if not all(col in available_columns for col in columns_to_load):
            print(f"One or more required columns ({columns_to_load}) not in the file.")
            return None, None

        data_df = parquet_file.read(columns=columns_to_load).to_pandas()
        
        if data_df is None:
            return None, None

        train_df = data_df[data_df['CaseID'].isin(config['train_caseIDs'])].copy()
        test_df = data_df[data_df['CaseID'].isin(config['test_caseIDs'])].copy()
        
        return train_df, test_df
    except Exception as e:
        print(f"An error occurred during data loading: {e}")
        return None, None

def plot_wear_class_distribution_interactive(wear_th):
    """
    Plots the distribution of 'wear_class' for train and test sets for each experiment
    based on the provided wear_th.
    """
    if not experiment_data:
        print("No experiment data to display.")
        return

    num_results = len(experiment_data)
    cols = 4  # 2 plots (train/test) for 2 experiments = 4 plots per row
    rows = (num_results + 1) // 2  # Ceiling division to get enough rows
    
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows), squeeze=False)

    for i, exp_data in enumerate(experiment_data):
        config = exp_data['config']
        train_df, test_df = load_wear_data_for_exp(config)

        row_idx = i // 2
        col_offset = (i % 2) * 2
        
        ax_train = axes[row_idx, col_offset]
        ax_test = axes[row_idx, col_offset + 1]

        if train_df is not None and not train_df.empty:
            train_df = set_wear_th(train_df, wear_th, config['wearColumnName'])
            sns.countplot(data=train_df, x='wear_class', ax=ax_train)
            ax_train.set_title(f"Train: {config['train_caseIDs']}")
            ax_train.set_xlabel("Wear Class")
            ax_train.set_ylabel("Count")
        else:
            ax_train.set_title(f"Train: {config['train_caseIDs']} (No data)")
            ax_train.set_visible(False)

        if test_df is not None and not test_df.empty:
            test_df = set_wear_th(test_df, wear_th, config['wearColumnName'])
            sns.countplot(data=test_df, x='wear_class', ax=ax_test)
            ax_test.set_title(f"Test: {config['test_caseIDs']}")
            ax_test.set_xlabel("Wear Class")
            ax_test.set_ylabel("Count")
        else:
            ax_test.set_title(f"Test: {config['test_caseIDs']} (No data)")
            ax_test.set_visible(False)

    # Hide any unused subplots
    for i in range(num_results * 2, rows * cols):
        axes.flatten()[i].set_visible(False)

    plt.suptitle(f'Distribution of Wear Class for Threshold: {wear_th:.2f}', fontsize=16, y=1.02)
    plt.tight_layout(rect=[0, 0, 1, 0.98]) # Adjust layout to make space for suptitle
    plt.show()

# Create an interactive slider for the wear threshold
interact(
    plot_wear_class_distribution_interactive,
    wear_th=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.5, description='Wear Threshold')
);


## 11. Autoencoder Anomaly Detection Evaluation

In [ ]:

# This cell evaluates an autoencoder model. It calculates the reconstruction error
# and then uses an interactive slider to set a threshold for anomaly detection.

# Check if there is experiment data and if the model is an autoencoder
if 'autoenc' in output_parent_dir and not results_df.empty:
    # The 'results_df' now contains the reconstruction errors and true labels
    # Select the first experiment as an example
    autoenc_results = results_df.iloc[0]
    
    recon_error = autoenc_results['reconstruction_error']
    y_true = autoenc_results['y_true']
    
    # Get the true labels for worn and non-worn tools
    is_worn = y_true == 1
    
    # Create a summary plot of reconstruction errors
    plt.figure(figsize=(12, 6))
    plt.hist(recon_error[~is_worn], bins=50, alpha=0.5, label='Healthy (Class 0)')
    plt.hist(recon_error[is_worn], bins=50, alpha=0.5, label='Worn (Class 1)')
    plt.xlabel("Reconstruction Error (MAE)")
    plt.ylabel("Number of Samples")
    plt.title("Distribution of Reconstruction Errors")
    plt.legend()
    plt.show()

    def evaluate_with_threshold(threshold):
        # Predict anomalies based on the threshold and convert to numpy
        y_pred = tf.cast(recon_error > threshold, tf.int32).numpy()
        
        # Calculate metrics
        accuracy = np.mean(y_pred == y_true)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        print(f"Threshold: {threshold:.4f}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")
        
        # Plot confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(cmap=plt.cm.Blues)
        plt.title(f'Confusion Matrix for Threshold: {threshold:.4f}')
        plt.show()

    # Interactive slider for the threshold
    interact(
        evaluate_with_threshold,
        threshold=FloatSlider(min=0.0, max=np.max(recon_error), step=0.01, value=0.1, description='Recon. Threshold')
    );

else:
    print("This evaluation is designed for autoencoder models.")
    print("Please set 'output_parent_dir' to an autoencoder experiment directory, e.g., '.../phm_milling_autoenc'.")


